In [ ]:
!pip install tensorflow
!pip install matplotlib numpy pillow

In [ ]:
import os
import warnings
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

# 1. SILENCE SYSTEM MESSAGES
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
warnings.filterwarnings('ignore')

# --- 2. FULL LEVEL 4 SETTINGS ---
# 313 steps * 32 batch size = 10,016 images processed per epoch
EPOCHS = 100            
STEPS_PER_EPOCH = 313   
IMG_SIZE = (150, 150)

# --- 3. THE "CLEAN CONSOLE" CALLBACK ---
class CleanProgress(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        # Overwrites the same line to keep the console clean
        print(f"\r🚀 Training Progress: Epoch {epoch+1}/{self.params['epochs']} "
              f"| Loss: {logs['loss']:.4f} | Acc: {logs['accuracy']:.4f} "
              f"| Val Acc: {logs['val_accuracy']:.4f}", end="")

# --- 4. DATA & MODEL ---
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
train_gen = train_datagen.flow_from_directory('Bone Fracture', target_size=IMG_SIZE, batch_size=32, class_mode='binary', subset='training')
val_gen = train_datagen.flow_from_directory('Bone Fracture', target_size=IMG_SIZE, batch_size=32, class_mode='binary', subset='validation', shuffle=False)

model_1 = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
model_1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# --- 5. TRAINING (With 10k Image Settings) ---
print(f"Model 1 Baseline: Processing {STEPS_PER_EPOCH * 32} images over {EPOCHS} epochs...")
model_1.fit(
    train_gen, 
    steps_per_epoch=STEPS_PER_EPOCH, 
    epochs=EPOCHS, 
    validation_data=val_gen,
    validation_steps=50,        # Increased for better statistical validation
    verbose=0,                  # Clean console mode
    callbacks=[CleanProgress()] 
)
print("\n✅ Training Complete. Generating Visuals...")

# --- 6. INDEPENDENT HIGH-RES VISUALS ---
def generate_final_visuals(model, val_gen):
    val_gen.reset()
    # Evaluating on a larger slice of validation data for the final report
    preds_prob = model.predict(val_gen, steps=50, verbose=0) 
    y_pred = (preds_prob > 0.5).astype(int).reshape(-1)
    y_true = val_gen.classes[:len(y_pred)]
    
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    
    # Quantitative Metrics
    acc = (tp + tn) / len(y_true) if len(y_true) > 0 else 0
    pre = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    # WINDOW 1: THE HEATMAP & COMPLETE METRIC CARD
    fig1, (ax_cm, ax_card) = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={'width_ratios': [1.2, 1]})
    
    # Heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax_cm, cbar=False,
                xticklabels=['Predicted Normal', 'Predicted Fracture'], 
                yticklabels=['Actual Normal', 'Actual Fracture'],
                annot_kws={"size": 26, "weight": "bold"})
    ax_cm.set_title('Confusion Matrix', fontsize=22, pad=20)

    # Metric Card (Big Four)
    ax_card.axis('off')
    ax_card.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, color='#f8f9fa', ec='#dee2e6', lw=2, transform=ax_card.transAxes))
    ax_card.text(0.5, 0.85, 'Metric Card', ha='center', fontsize=30, fontweight='bold')
    
    metrics_data = [
        (f'Accuracy: {acc:.1%}', 'navy'),
        (f'Precision: {pre:.1%}', 'darkred'),
        (f'Recall: {rec:.1%}', 'darkgreen'),
        (f'F1-Score: {f1:.1%}', '#8e44ad')
    ]
    
    y_pos = 0.65
    for text, color in metrics_data:
        ax_card.text(0.5, y_pos, text, ha='center', fontsize=24, color=color, fontweight='bold')
        y_pos -= 0.12
        
    plt.tight_layout()
    plt.show()

    # WINDOW 2: DETAILED DIAGNOSIS (LARGE BAR CHART)
    plt.figure(figsize=(16, 10))
    categories = ['True Negative\n(Correct Normal)', 'True Positive\n(Correct Fracture)', 
                  'Type 1 Error\n(False Positive)', 'Type 2 Error\n(False Negative)']
    counts = [tn, tp, fp, fn]
    colors = ['#2ecc71', '#27ae60', '#f1c40f', '#e74c3c']
    
    bars = plt.bar(categories, counts, color=colors, edgecolor='black', alpha=0.85)
    
    # Buffer space so labels don't hit the top
    plt.ylim(0, max(counts) * 1.3) 
    
    plt.title('Detailed Diagnosis Breakdown', fontsize=26, fontweight='bold', pad=30)
    plt.ylabel('Patient Count', fontsize=18)
    plt.xticks(fontsize=14, fontweight='bold')
    
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + (max(counts)*0.01),
                 f'{int(height)}', ha='center', va='bottom', fontsize=22, fontweight='bold')

    plt.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

# Run visuals
generate_final_visuals(model_1, val_gen)